# 00 — Rodar o pipeline (Colab ou local)

Este notebook roda o **pipeline completo** pela CLI, na ordem certa. É pensado para o Colab
(GPU T4), mas também funciona com o kernel local `.venv`.

- No **Colab**, a célula de ambiente monta o Drive, clona o repositório, instala as dependências,
  baixa o dataset e lê ou calcula as features. `outputs/` fica ligado a `MyDrive/pathoshift/outputs`.
- **Localmente**, prefira o terminal (`python pathoshift.py ...`, veja o README). Aqui, por segurança,
  nada é treinado enquanto `RODAR_PIPELINE = False`.

In [ ]:
# ===== Célula de ambiente (igual em todos os notebooks) =====
# Local (kernel .venv): usa o que já está em data/ e outputs/; não instala nem baixa nada.
# Colab (extensão do VS Code ou navegador): monta o Drive, clona/atualiza o repositório,
# instala as dependências e prepara o que este notebook precisa. Preencha REPO_URL para o Colab.
REPO_URL = "https://github.com/SEU_USUARIO/pathoshift.git"
BRANCH = "main"

import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/pathoshift")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ambiente import preparar
preparar(ROOT, precisa_dados=True, precisa_features=True, precisa_resultados=False)

In [ ]:
import subprocess, sys

RODAR_PIPELINE = False   # mude para True para treinar tudo (CV + modelos finais + predições + métricas)

def rodar(*args):
    print("$ python pathoshift.py", *args)
    subprocess.run([sys.executable, "pathoshift.py", *args], check=True)

rodar("describe")

In [ ]:
if RODAR_PIPELINE:
    rodar("train", "--model", "logreg")
    rodar("train", "--model", "cnn")
    rodar("train", "--model", "logreg", "--final")
    rodar("train", "--model", "cnn", "--final")
    rodar("predict")
    rodar("evaluate")
else:
    print("RODAR_PIPELINE = False: nada foi treinado.")

## Teste externo (centro 2)
**Já foi executado uma vez** (veja `outputs/run_info.json`). O comando recusa uma segunda execução
sem `--forcar`. Não rode de novo só para "ver se melhora".